In [0]:
def walk(path, depth=0):
    for f in dbutils.fs.ls(path):
        print("  " * depth + f.name, f.size if not f.isDir() else "")
        if f.isDir():
            walk(f.path, depth + 1)

walk("/Volumes/workspace/bronze/landing")

fakestore_api/ 
  products/ 
    25/ 
      11/ 
        2026/ 
          products.json 10662
inventory_csv/ 
  inventory_20261118.csv 4779
  inventory_20261119.csv 4770
  inventory_20261120.csv 4773
  inventory_20261121.csv 4772
  inventory_20261122.csv 4777
  inventory_20261123.csv 4760
  inventory_20261124.csv 4780
sqlserver/ 
  customers/ 
    2026/ 
      11/ 
        25/ 
          customers.parquet 7444
  order_items/ 
    2026/ 
      11/ 
        25/ 
          order_items.parquet 115831
  orders/ 
    2026/ 
      11/ 
        25/ 
          orders.parquet 44885
  stores/ 
    2026/ 
      11/ 
        25/ 
          stores.parquet 1604


In [0]:
base = "/Volumes/workspace/bronze/landing"

expected = {"stores": 10, "customers": 200, "orders": 4000, "order_items": 12001}
for t, exp in expected.items():
    n = (spark.read.option("recursiveFileLookup", "true")
         .parquet(f"{base}/sqlserver/{t}").count())
    print(f"{t:12} rows={n:>6}  expected={exp:>6}  {'OK' if n == exp else 'MISMATCH'}")

products = (spark.read.option("recursiveFileLookup", "true")
            .option("multiLine", "true")
            .json(f"{base}/fakestore_api/products"))
print("products rows:", products.count())

inv = (spark.read.option("header", "true")
       .csv(f"{base}/inventory_csv/*.csv"))
print("inventory rows:", inv.count(), "from 7 files")

stores       rows=    10  expected=    10  OK
customers    rows=   200  expected=   200  OK
orders       rows=  4000  expected=  4000  OK
order_items  rows= 12001  expected= 12001  OK
products rows: 1
inventory rows: 1428 from 7 files


In [0]:
products = (spark.read.option("recursiveFileLookup", "true")
            .option("multiLine", "true")
            .json("/Volumes/workspace/bronze/landing/fakestore_api/products"))
products.printSchema()

# Also show the raw start of the file
raw = spark.read.option("recursiveFileLookup", "true").text(
    "/Volumes/workspace/bronze/landing/fakestore_api/products", wholetext=True)
print(raw.first()[0][:600])

root
 |-- category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- id: long (nullable = true)
 |-- image: string (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: struct (nullable = true)
 |    |-- count: long (nullable = true)
 |    |-- rate: double (nullable = true)
 |-- title: string (nullable = true)

﻿{"id":1,"title":"Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops","price":109.95,"description":"Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday","category":"men's clothing","image":"https://fakestoreapi.com/img/81fPKd-2AYL._AC_SL1500_t.png","rating":{"rate":3.9,"count":120}}
{"id":2,"title":"Mens Casual Premium Slim Fit T-Shirts ","price":22.3,"description":"Slim-fitting style, contrast raglan long sleeve, three-button henley placket, light weight & soft fabric for breathable and comfortable wearing. An


In [0]:
path = "/Volumes/workspace/bronze/landing/fakestore_api/products"

products = (spark.read
            .option("recursiveFileLookup", "true")
            .json(path))          # no multiLine: one product per line

print("rows:", products.count())
products.select("id", "title", "price", "category", "rating.rate", "rating.count").show(5, truncate=40)
print("columns:", products.columns)

rows: 20
+---+----------------------------------------+------+--------------+----+-----+
| id|                                   title| price|      category|rate|count|
+---+----------------------------------------+------+--------------+----+-----+
|  1|Fjallraven - Foldsack No. 1 Backpack,...|109.95|men's clothing| 3.9|  120|
|  2|  Mens Casual Premium Slim Fit T-Shirts |  22.3|men's clothing| 4.1|  259|
|  3|                      Mens Cotton Jacket| 55.99|men's clothing| 4.7|  500|
|  4|                    Mens Casual Slim Fit| 15.99|men's clothing| 2.1|  430|
|  5|John Hardy Women's Legends Naga Gold ...| 695.0|      jewelery| 4.6|  400|
+---+----------------------------------------+------+--------------+----+-----+
only showing top 5 rows
columns: ['category', 'description', 'id', 'image', 'price', 'rating', 'title']
